# UK Flood Risk Exposure Analysis using Python #

### Objective: 
An analysis of UK Postcode level flood risk data to demonstrate catastrophe risk and exposure management concepts, including: flood hazard analysis, exposure aggregation, concentration analysis and simulated loss estimation.  

### Tools used: 
- Python
- Jupyter Notebook
- Pandas
- Numpy
- Matplotlib

### Dataset - Open Flood Risk by Postcode Dataset 
The dataset contains postcode level flood risk classifications derived from the Enviornment Agency's 'Risk of Flooding from Rivers and Sea' dataset and Open Postcode Geo. Flood risk is categorised into four bands (High, Medium, Low, and Very Low) and includes postcode-level geographic coordinates. The working dataset used for this project was accessed via Kaggle. 

### Limitations
Real insurance portfolio and exposure data are proprietary and not publicly available. Due to this, insured values were simulated for the purposes of this project to demonstrate exposure aggregation, concentration analysis and catastrophe-style loss estimation techniques.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
fr=pd.read_csv('/Users/sam_bradberry/Documents/Data Science & Python learning/Data to work with/open_flood_risk_by_postcode.csv')
fr.head()

### Data Quality Checks

In [ ]:
fr.info

In [ ]:
fr.columns

In [ ]:
fr.shape

### Data Cleaning
- Data imported incorrectly with the first row interpreted as headers.
- Reloading dataset with header=None, renaming columns and preparing the dataset for analysis.

In [ ]:
fr.columns=['Index','Postcode','FID','PROB_4BAND','SUITABILITY','PUB_DATE','Risk_For_Insurance_SOP','Easting','Northing','Latitude','Longitude']

fr.head()

In [ ]:
fr=fr.iloc[:,1:]

In [ ]:
fr.shape

In [ ]:
fr.head()

In [ ]:
fr.info

### Flood Hazard Distribution

In [ ]:
fr.describe(include='all')

In [ ]:
fr['PROB_4BAND'].value_counts()

### Exposure Aggregation and Modelling
As insurance portfolio data is not publicly available, simulated insured property values are randomly assigned (£75,000 - £1,000,000) to demonstrate exposure management concepts.

In [ ]:
fr['PROB_4BAND'].value_counts().plot(kind='bar')

plt.title('Flood Risk Category Distribution')
plt.xlabel('Flood Risk')
plt.ylabel('Number of Postcodes')

In [ ]:
risk_order=['Very Low','Low','Medium','High']
risk_counts=fr['PROB_4BAND'].value_counts().reindex(risk_order)

risk_counts.plot(kind='bar')

plt.title('Flood Risk Category Distribution')
plt.xlabel('Flood Risk')
plt.ylabel('Number of Postcodes')

In [ ]:
fr['Insured_Value']=np.random.randint(75000,1000000,size=len(fr))

In [ ]:
risk_exposure=fr.groupby('PROB_4BAND')['Insured_Value'].sum()

print(risk_exposure)

In [ ]:
risk_order=['Very Low','Low','Medium','High']
risk_exposure=fr.groupby('PROB_4BAND')['Insured_Value'].sum().reindex(risk_order)

risk_exposure.plot(kind='bar')

plt.title('Total Insured Exposure by Flood Risk')
plt.ylabel('Insured Value(£)')
plt.xlabel('Flood Risk')

### Flood Loss Model
This loss model is applied using assumed damage ratios for each flood risk category (High to Very Low - 10%, 5%, 2% and 0.5% respectively). These assumptions are for the purposes of this workbook only and are not intended to represent real catastrophe model values.

In [ ]:
loss_values={
    'High':0.10,
    'Medium':0.05,
    'Low':0.02,
    'Very Low':0.005}

In [ ]:
fr['Loss']=fr['PROB_4BAND'].map(loss_values)

In [ ]:
fr['Estimated_Loss']=fr['Insured_Value']*fr['Loss']

In [ ]:
estimated_loss=fr.groupby('PROB_4BAND')['Estimated_Loss'].sum()

print(estimated_loss)

In [ ]:
risk_order=['Very Low','Low','Medium','High']
estimated_loss=estimated_loss.reindex(risk_order)

estimated_loss.plot(kind='bar')

plt.title('Total Estimated Flood Loss by Risk Category')
plt.xlabel('Flood Risk')
plt.ylabel('Estimated Loss(£)')

In [ ]:
estimated_loss.plot(kind='pie',autopct='%1.2f%%')

plt.title('Share of Total Estimated Loss by Flood Risk')
plt.ylabel('')

In [ ]:
estimated_loss.plot(kind='pie',autopct='%1.0f%%')

plt.title('Share of Total Estimated Loss by Flood Risk')
plt.ylabel('')

### Exposure Concentrations
The following analysis identifies the postcode areas with the highest and lowest levels of insured exposure.

In [ ]:
fr['Postcode_Area']=fr['Postcode'].str.split().str[0]

In [ ]:
area_exposure=fr.groupby('Postcode_Area')['Insured_Value'].sum()

In [ ]:
top10=area_exposure.sort_values(ascending=False).head(10)

top10.plot(kind='barh')

plt.title('Top 10 Postcode Areas by Insured Exposure')
plt.ylabel('Postcode Areas')
plt.xlabel('Insured Value (£)')

The above postcode areas represent the largest concentrations of insured exposure to flood risk. These postcodes may warrant further review for managing potential loss.

In [ ]:
top10=area_exposure.sort_values(ascending=True).head(10)

top10.plot(kind='barh')

plt.title('Bottom 10 Postcode Areas by Insured Exposure')
plt.ylabel('Postcode Areas')
plt.xlabel('Insured Value (£)')

The above postcode areas contribute the least to the overall portfolio exposure. These postcodes therefore have a limited impact on loss potential.

### Full Distribution of Insurance Exposure using bins

In [ ]:
area_exposure.hist(bins=10)

plt.title('Distribution of Aggregated Exposure by Postcode Areas')
plt.xlabel('Total Insured Exposure (£)')
plt.ylabel('Frequency')

By narrowing the histogram bins to 10, the distribution of aggregated insured exposure becomes more clearly observable. The results indicate that the majority of postcode areas within the simulated portfolio contain relatively lower levels of aggregated insured exposure, while the frequency of postcode areas declines significantly as total exposure increases. This interpretation suggests a positively skewed exposure distribution, whereby a relatively small number of postcode areas contain disproportionately large concentrations of insured value. These higher concentration areas may represent greater risk during severe flood events, as shown when we combine this with the data of estimated losses- High Risk postcode areas, whilst distributed in a clear minority, present a disproportionately high contribution to potential loss.

### Conclusions

Key observations:
- The majority of UK postcodes fall within the low flood risk category.
- High-risk postcode areas contribute disproportionately to potential loss.

Future enhancements:
- Event-based flood scenarios.
- Geospatial Mapping.
- Probable Maximum Loss estimations
- Annual Average Loss calculations

Limitations:
- Simulated insured values.
- Loss assumptions were simplified.
- No event or catastrophe model was applied.